<a href="https://colab.research.google.com/github/canurdon/SnowLine/blob/main/sentinel2_snow_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import ee
import geemap
import xml.etree.ElementTree as ET
import datetime
dem = ee.Image('USGS/SRTMGL1_003')

ee.Authenticate()
ee.Initialize(project='snowline-491111')

In [38]:
!pip install earthengine-api geemap -q

In [39]:
#create Tantalus bounding box
aoi = ee.Geometry.Rectangle([-123.38, 49.75, -123.10, 49.95])

#Verify area loaded correctly
print('area of interest defined')
print('Bounds:', aoi.bounds().getInfo())

area of interest defined
Bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-123.38000000000001, 49.74999999999998], [-123.10000000000001, 49.74999999999998], [-123.10000000000001, 49.95008424772833], [-123.38000000000001, 49.95008424772833], [-123.38000000000001, 49.74999999999998]]]}


In [40]:
# Dynamic date range — most recent 60 days
end = datetime.datetime.now()
start = end - datetime.timedelta(days=60)

end_str = end.strftime('%Y-%m-%d')
start_str = start.strftime('%Y-%m-%d')

print(f'Searching: {start_str} to {end_str}')

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate(start_str, end_str) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
    .sort('system:time_start', False)

image = s2.first()

info = image.getInfo()
print('Image ID:', info['id'])
print('Date:', ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print('Cloud cover:', image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(), '%')

Searching: 2026-01-25 to 2026-03-26
Image ID: COPERNICUS/S2_SR_HARMONIZED/20260308T192139_20260308T192314_T10UDA
Date: 2026-03-08
Cloud cover: 29.023671 %


In [41]:
# Select the bands we need
# Band 3 = Green, Band 11 =SWIR
green = image.select('B3')
swir = image.select('B11')

# Compute NDSI
# Formula: (Green - SWIR) / (Green + SWIR)
ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')

#Threshhold at 0.55 - standard snow detection cutoff
snow_mask = ndsi.gt(0.55)

# Visualise on an interactive map
map1 = geemap.Map()
map1.centerObject(aoi, zoom=11)

# Add layers
map1.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map1.addLayer(
    ndsi,
    {'min': -1, 'max': 1, 'palette': ['black', 'white']},
    'NDSI'
)
map1.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask'
)

map1


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

## Create cloud mask
Avoids classifying cloud as snow.

In [42]:
# SCL identifies the type of terrain represented by each pixel
# Numbers we want to mask out:
# 3 – cloud shadow; 8 – cloud medium probability; 9 – cloud high probability; 10 – thin cirrus

# create variable for SCL
scl = image.select('SCL')

# create cloud mask: keep = 1; mask out = 0
cloud_mask = scl.neq(3) \
.And(scl.neq(8)) \
.And(scl.neq(9)) \
.And(scl.neq(10))

# Apply cloud mask to NDSI
ndsi_masked = ndsi.updateMask(cloud_mask)
snow_mask_masked = ndsi_masked.gt(0.45)

# Add to map
map2 = geemap.Map()
map2.centerObject(aoi, zoom=11)

map2.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map2.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask unmasked'
)
map2.addLayer(
    snow_mask_masked,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask masked'
)
map2.addLayer(
    cloud_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ffffff']},
    'Cloud mask'
)

map2

Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

## Snow line detection
Create a snow line detector based on the elevation of lowest high probability snow pixel

In [43]:
dem = ee.Image('USGS/SRTMGL1_003')

# Strict threshold on most recent clear image
strict_snow = ndsi.gt(0.6) \
    .And(image.select('B3').gt(1500)) \
    .And(cloud_mask) \
    .And(water_mask)

# 10th percentile elevation of confident snow pixels
snowline_result = dem.updateMask(strict_snow) \
    .reduceRegion(
        reducer=ee.Reducer.percentile([10]),
        geometry=aoi,
        scale=100,
        maxPixels=1e9
    )

snowline = snowline_result.get('elevation')
snowline_buffered = ee.Number(snowline).subtract(200)

print('Snowline elevation:', snowline.getInfo(), 'm')
print('Buffered snowline:', snowline_buffered.getInfo(), 'm')

Snowline elevation: 1043.3560413589364 m
Buffered snowline: 843.3560413589364 m


# Multiple Image Compilation
Using a compilation of images to get an mosaic that is resistant to cloud cover.


In [46]:
# Create 60 day window over which to assess each pixel
end_date = ee.Date(datetime.datetime.now())
start_date = end_date.advance(-60, 'day')

# Gather collection
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
.filterBounds(aoi) \
.filterDate(start_date, end_date)

# Create unction to mask clouds and compute NDSI for each image
def process_image(image):
    scl = image.select('SCL')

    cloud_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    water_mask = scl.neq(6)
    valid_mask = cloud_mask.And(water_mask)

    ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')
    ndsi_masked = ndsi.updateMask(valid_mask)

    timestamp = image.metadata('system:time_start').rename('timestamp').clip(aoi)

    return ndsi_masked.addBands(timestamp).clip(aoi) \
        .copyProperties(image, ['system:time_start'])

# Apply function to collection
processed = collection.map(process_image)

# Composite: most recent valid pixel
composite = processed.qualityMosaic('timestamp').clip(aoi)


print('composite band created')
print(' names:', composite.bandNames().getInfo())

composite band created
 names: ['NDSI', 'timestamp']


## Calculate snowline
Calculte the estimated snowline and use it to make a mask that excludes false positives at lower elevations.

In [47]:
# All tunable parameters in one place
NDSI_SNOW_THRESHOLD = 0.4
NDSI_STRICT_THRESHOLD = 0.6
B3_BRIGHTNESS_FLOOR = 1500
SNOWLINE_BUFFER_M = 200
STALENESS_FRESH = 3
STALENESS_MODERATE = 7

# Derive snowline from recent clear image
strict_snow = ndsi.gt(NDSI_STRICT_THRESHOLD) \
    .And(image.select('B3').gt(B3_BRIGHTNESS_FLOOR)) \
    .And(cloud_mask).And(water_mask)

snowline = dem.updateMask(strict_snow).reduceRegion(
    reducer=ee.Reducer.percentile([10]),
    geometry=aoi, scale=100, maxPixels=1e9
).get('elevation')
snowline_buffered = ee.Number(snowline).subtract(SNOWLINE_BUFFER_M)

# Apply interpretations to composite
ndsi_composite = composite.select('NDSI')

snow_binary = ndsi_composite.gt(NDSI_SNOW_THRESHOLD)
above_snowline = dem.gt(snowline_buffered)
snow_confident = snow_binary.And(above_snowline)

# Staleness
now_ms = ee.Date(datetime.datetime.now()).millis()
days_ago = composite.select('timestamp') \
    .subtract(now_ms).abs().divide(86400000).rename('days_ago')

# Package everything
output = snow_confident.rename('snow') \
    .addBands(ndsi_composite) \
    .addBands(above_snowline.rename('above_snowline')) \
    .addBands(days_ago)

In [15]:
# Visualise the composite
map3 = geemap.Map()
map3.centerObject(aoi, zoom=11)

# Extract the snow band
snow_composite = composite.select('snow')

# Extract timestamp and convert to days ago
now_ms = ee.Date(datetime.datetime.now()).millis()
days_ago = composite.select('timestamp') \
.subtract(now_ms) \
.abs() \
.divide(86400000) \
.rename('days_ago')

map3.addLayer(
    snow_composite,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow composite'
)

map3.addLayer(
    days_ago,
    {'min': 0, 'max': 60, 'palette': ['00ff00', 'ffff00', 'ff0000']},
    'Pixel age (days)'
)
map3.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'true colour'
)

map3

Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

# GPS Track Functionality
These cells allow the user to upload a GPS track and analyse the route against the terrain layers.

In [16]:
# Parse GPX file
def parse_gpx(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()

    # Handle GPX namespace
    ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}

    coords = []
    elevations = []

    for trkpt in root.findall('.//gpx:trkpt', ns):
        lat = float(trkpt.get('lat'))
        lon = float(trkpt.get('lon'))
        ele = trkpt.find('gpx:ele', ns)
        elev = float(ele.text) if ele is not None else 0

        coords.append([lon, lat])
        elevations.append(elev)

    return coords, elevations

# Load the route
coords, elevations = parse_gpx('/content/tantalusFKT_2024.gpx')

print(f'Track points: {len(coords)}')
print(f'Start: {coords[0]}')
print(f'End: {coords[-1]}')
print(f'Min elevation: {min(elevations)}m')
print(f'Max elevation: {max(elevations)}m')

Track points: 21699
Start: [-123.3229366, 49.9105733]
End: [-123.2026349, 49.79482]
Min elevation: 101.0m
Max elevation: 2601.0m


## Shrink the data
Coros records a position every second, which leads to significant compute burdens. Taking every 50th point reduces burden while retaining sufficient resolution for our 10m satelite images.

In [17]:
# Simplify route — take every 50th point
# 21699 / 50 = ~434 points, plenty for 10m resolution
simplified = coords[::50]
print(f'Simplified to {len(simplified)} points')

# Create Earth Engine line geometry
route = ee.Geometry.LineString(simplified)

# Display on map with new AOI
map4 = geemap.Map()
map4.centerObject(route, zoom=11)

# Add snow composite
map4.addLayer(
    composite.select('snow'),
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow composite'
)

# Add route
map4.addLayer(
    ee.Image().paint(route, 1, 2),
    {'palette': ['ff0000']},
    'Tantalus Traverse'
)

map4

Simplified to 434 points


Map(center=[49.837876508085344, -123.31442795002913], controls=(WidgetControl(options=['position', 'transparen…

## Analyse snow along route
Calculate the percentage snow coverage along the route and then analyse a sample of points along route for more detailed analysis

In [25]:
# Sample snow values along the route
# What is the mean snow value for all 10m pixels that intersect the route geometry?
route_snow = composite.select('snow').reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=route,
    scale=10,
    maxPixels=1e9
)

print('Mean snow coverage along route:', route_snow.getInfo())

# Sample individual points along route for detailed analysis
sampled = composite.select('snow').sample(
    region=route,
    scale=10,
    numPixels=1000,
    geometries=True
)

# Count snow vs non-snow points
snow_points = sampled.filter(ee.Filter.eq('snow', 1))
clear_points = sampled.filter(ee.Filter.eq('snow', 0))

total = sampled.size().getInfo()
snow_count = snow_points.size().getInfo()
clear_count = clear_points.size().getInfo()

print(f'Total sampled points: {total}')
print(f'Snow points: {snow_count}')
print(f'Clear points: {clear_count}')
print(f'Snow coverage: {round(snow_count/total*100, 1)}%')

Mean snow coverage along route: {'snow': 0.693326358483401}
Total sampled points: 3489
Snow points: 2406
Clear points: 1083
Snow coverage: 69.0%


##Snow Segments
Find segments of the route that cross snow using samples 1000 points on route.

In [29]:
# Get coordinates and snow values for all sampled points
features = sampled.getInfo()['features']

# Extract into a simple list
points = []
for f in features:
    coords = f['geometry']['coordinates']
    snow = f['properties']['snow']
    points.append({
        'lon': coords[0],
        'lat': coords[1],
        'snow': snow
    })

# Sort by longitude as rough proxy for position along route
# Sigurd is west (-123.32), Alpha is east (-123.20)
points.sort(key=lambda x: x['lon'])

# Identify snow/clear transitions
segments = []
current_state = points[0]['snow']
segment_start = points[0]
segment_points = [points[0]]

for p in points[1:]:
    if p['snow'] != current_state:
        segments.append({
            'type': 'snow' if current_state == 1 else 'clear',
            'start_lon': segment_start['lon'],
            'start_lat': segment_start['lat'],
            'end_lon': p['lon'],
            'end_lat': p['lat'],
            'point_count': len(segment_points)
        })
        current_state = p['snow']
        segment_start = p
        segment_points = [p]
    else:
        segment_points.append(p)

# Print snow segments
# print('Snow crossing segments:')
# snow_segs = [s for s in segments if s['type'] == 'snow']
# for i, s in enumerate(snow_segs):
#   print(f'  Segment {i+1}: {s["start_lon"]:.4f} to {s["end_lon"]:.4f} ({s["point_count"]} points)')

In [30]:
def haversine(coord1, coord2):
    """Distance in metres between two [lon, lat] points"""
    R = 6371000
    lat1, lat2 = math.radians(coord1[1]), math.radians(coord2[1])
    dlat = math.radians(coord2[1] - coord1[1])
    dlon = math.radians(coord2[0] - coord1[0])
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Build list of points with cumulative distance along route
# Use original simplified coords as the route backbone
route_with_dist = []
cumulative_dist = 0
for i, coord in enumerate(simplified):
    if i > 0:
        cumulative_dist += haversine(simplified[i-1], coord)
    route_with_dist.append({
        'lon': coord[0],
        'lat': coord[1],
        'dist_m': cumulative_dist
    })

total_route_km = cumulative_dist / 1000
print(f'Total route length: {total_route_km:.1f} km')

# For each sampled point, find nearest route point and get its distance
def nearest_route_dist(point, route):
    min_dist = float('inf')
    nearest = None
    for rp in route:
        d = haversine([point['lon'], point['lat']], [rp['lon'], rp['lat']])
        if d < min_dist:
            min_dist = d
            nearest = rp
    return nearest['dist_m']

# Tag each sample point with distance along route
print('Calculating distances along route...')
for p in points:
    p['route_dist_m'] = nearest_route_dist(p, route_with_dist)

# Sort by distance along route
points.sort(key=lambda x: x['route_dist_m'])
print('Done')


Total route length: 29.5 km
Calculating distances along route...
Done


In [31]:
# Rebuild segments using route distance
MIN_SEGMENT_M = 100  # ignore snow patches shorter than 100m

segments_by_dist = []
current_state = points[0]['snow']
segment_start = points[0]
segment_points = [points[0]]

for p in points[1:]:
    if p['snow'] != current_state:
        dist_m = segment_points[-1]['route_dist_m'] - segment_start['route_dist_m']
        segments_by_dist.append({
            'type': 'snow' if current_state == 1 else 'clear',
            'start_m': segment_start['route_dist_m'],
            'end_m': segment_points[-1]['route_dist_m'],
            'start_lon': segment_start['lon'],
            'start_lat': segment_start['lat'],
            'end_lon': segment_points[-1]['lon'],
            'end_lat': segment_points[-1]['lat'],
            'length_m': dist_m,
            'point_count': len(segment_points)
        })
        current_state = p['snow']
        segment_start = p
        segment_points = [p]
    else:
        segment_points.append(p)

# Merge short segments into neighbours
def merge_short(segments, min_m):
    merged = []
    for seg in segments:
        if seg['length_m'] < min_m and merged:
            prev = merged[-1]
            prev['end_m'] = seg['end_m']
            prev['end_lon'] = seg['end_lon']
            prev['end_lat'] = seg['end_lat']
            prev['length_m'] = prev['end_m'] - prev['start_m']
            prev['point_count'] += seg['point_count']
        else:
            merged.append(dict(seg))
    return merged

# Run merge twice to catch cascading short segments
merged = merge_short(segments_by_dist, MIN_SEGMENT_M)
merged = merge_short(merged, MIN_SEGMENT_M)

# Print meaningful snow crossings
snow_segs = [s for s in merged if s['type'] == 'snow']
print(f'Snow crossings: {len(snow_segs)}')
print()
for i, s in enumerate(snow_segs):
    print(f'Crossing {i+1}:')
    print(f'  Start: {s["start_m"]/1000:.1f}km')
    print(f'  End:   {s["end_m"]/1000:.1f}km')
    print(f'  Length: {s["length_m"]/1000:.1f}km')
    print()

Snow crossings: 18

Crossing 1:
  Start: 4.6km
  End:   4.8km
  Length: 0.2km

Crossing 2:
  Start: 5.3km
  End:   6.0km
  Length: 0.6km

Crossing 3:
  Start: 6.0km
  End:   6.2km
  Length: 0.3km

Crossing 4:
  Start: 6.3km
  End:   6.6km
  Length: 0.3km

Crossing 5:
  Start: 6.6km
  End:   7.1km
  Length: 0.4km

Crossing 6:
  Start: 7.1km
  End:   7.3km
  Length: 0.2km

Crossing 7:
  Start: 7.3km
  End:   11.2km
  Length: 3.9km

Crossing 8:
  Start: 11.5km
  End:   12.5km
  Length: 1.0km

Crossing 9:
  Start: 12.5km
  End:   14.6km
  Length: 2.1km

Crossing 10:
  Start: 14.6km
  End:   15.4km
  Length: 0.8km

Crossing 11:
  Start: 15.4km
  End:   23.3km
  Length: 7.9km

Crossing 12:
  Start: 23.3km
  End:   24.4km
  Length: 1.0km

Crossing 13:
  Start: 24.8km
  End:   25.0km
  Length: 0.1km

Crossing 14:
  Start: 25.0km
  End:   25.1km
  Length: 0.1km

Crossing 15:
  Start: 25.1km
  End:   25.2km
  Length: 0.2km

Crossing 16:
  Start: 25.2km
  End:   25.7km
  Length: 0.5km

Crossing 1

In [32]:
# SCL-based snow mask — ESA's own classification
# Value 11 = snow/ice in the Scene Classification Layer
scl_full = image.select('SCL')
scl_snow_mask = scl_full.eq(11)

# Compare against NDSI mask
# Four possible outcomes per pixel:
# Both agree snow     = true positive
# Both agree no snow  = true negative
# NDSI says snow, SCL says no = false positive (rock falsely detected)
# SCL says snow, NDSI says no = false negative (snow missed)

true_positive  = snow_mask.And(scl_snow_mask)
true_negative  = snow_mask.Not().And(scl_snow_mask.Not())
false_positive = snow_mask.And(scl_snow_mask.Not())
false_negative = snow_mask.Not().And(scl_snow_mask)

# Count pixels in each category over AOI
def count_pixels(img):
    return img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=20,
        maxPixels=1e9
    ).values().get(0)

tp = count_pixels(true_positive).getInfo()
tn = count_pixels(true_negative).getInfo()
fp = count_pixels(false_positive).getInfo()
fn = count_pixels(false_negative).getInfo()

total = tp + tn + fp + fn

print(f'True positives  (both agree snow):     {tp:>8.0f} pixels ({tp/total*100:.1f}%)')
print(f'True negatives  (both agree no snow):  {tn:>8.0f} pixels ({tn/total*100:.1f}%)')
print(f'False positives (NDSI snow, SCL clear):{fp:>8.0f} pixels ({fp/total*100:.1f}%)')
print(f'False negatives (SCL snow, NDSI clear):{fn:>8.0f} pixels ({fn/total*100:.1f}%)')
print()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
accuracy  = (tp + tn) / total

print(f'Precision: {precision:.3f}  (of pixels NDSI calls snow, how many does SCL agree?)')
print(f'Recall:    {recall:.3f}  (of pixels SCL calls snow, how many does NDSI catch?)')
print(f'Accuracy:  {accuracy:.3f}  (overall agreement)')

# Visualise disagreements on map
map5 = geemap.Map()
map5.centerObject(aoi, zoom=11)

map5.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map5.addLayer(
    true_positive,
    {'min': 0, 'max': 1, 'palette': ['000000', '0000ff']},
    'True positive (both agree snow)'
)
map5.addLayer(
    false_positive,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ff0000']},
    'False positive (NDSI says snow, SCL disagrees)'
)
map5.addLayer(
    false_negative,
    {'min': 0, 'max': 1, 'palette': ['000000', '00ff00']},
    'False negative (SCL says snow, NDSI misses)'
)
map5.addLayer(
    scl_snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00ffff']},
    'SCL snow only'
)

map5

True positives  (both agree snow):       182646 pixels (16.3%)
True negatives  (both agree no snow):    737476 pixels (65.9%)
False positives (NDSI snow, SCL clear):  167352 pixels (15.0%)
False negatives (SCL snow, NDSI clear):   31522 pixels (2.8%)

Precision: 0.522  (of pixels NDSI calls snow, how many does SCL agree?)
Recall:    0.853  (of pixels SCL calls snow, how many does NDSI catch?)
Accuracy:  0.822  (overall agreement)


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

In [33]:
dem = ee.Image('USGS/SRTMGL1_003')

# Define SCL and masks for this scope
scl = image.select('SCL')
cloud_mask = scl.neq(3) \
    .And(scl.neq(8)) \
    .And(scl.neq(9)) \
    .And(scl.neq(10))
water_mask = scl.neq(6) # Water mask – prevents bodies of water being identified as snow

# Strict threshold on most recent clear image
strict_snow = ndsi.gt(0.6) \
    .And(image.select('B3').gt(1500)) \
    .And(cloud_mask) \
    .And(water_mask)

# 10th percentile elevation of confident snow pixels
snowline_result = dem.updateMask(strict_snow) \
    .reduceRegion(
        reducer=ee.Reducer.percentile([10]),
        geometry=aoi,
        scale=100,
        maxPixels=1e9
    )

snowline = snowline_result.get('elevation')
snowline_buffered = ee.Number(snowline).subtract(200)

print('Snowline elevation:', snowline.getInfo(), 'm')
print('Buffered snowline:', snowline_buffered.getInfo(), 'm')

Snowline elevation: 1043.3560413589364 m
Buffered snowline: 843.3560413589364 m
